# 🏪 Store Sales - Time Series Forecasting

## 🎯 商业背景 (Business Context)
> 你是厄瓜多尔最大连锁超市 **Favorita** 的 Senior Data Analyst。
> 公司有 **54 家门店**，销售 **33 个品类**的商品。
> 管理层需要你基于历史数据预测各**门店 × 品类**的每日销量，以优化库存与排班。

## 📦 数据集概览
| 文件 | 内容 | 关键字段 |
| :--- | :--- | :--- |
| `train.csv` | 历史销量 (3M行) | `date`, `store_nbr`, `family`, `sales`, `onpromotion` |
| `stores.csv` | 门店信息 (54家) | `store_nbr`, `city`, `state`, `type`, `cluster` |
| `oil.csv` | 日油价 (外部变量) | `date`, `dcoilwtico` |
| `holidays_events.csv` | 节假日 (350条) | `date`, `type`, `locale`, `description` |
| `transactions.csv` | 日交易量 | `date`, `store_nbr`, `transactions` |

## 🛣️ 路线图 (Roadmap)
1. **🧹 Data Prep**: 加载 & 合并多表 → 时间解析 → 缺失值处理
2. **📊 EDA**: 按门店/品类/时间的销量分析 → 促销效果 → 油价相关性
3. **🔧 Feature Eng**: 日期特征 + **Per-Category Lag/Diff/Rolling** + 促销 + 节假日 + 油价
4. **🔮 Modeling**: XGBoost (带 Early Stopping + Feature Importance 检查)
5. **📉 Evaluation**: 分层评估 (Normal vs Extreme) + 按品类 M[目标业务] + 残差分析
6. **💡 Insights**: 哪些品类最难预测？促销对预测的影响？

---

## ⛽️ 函数加油站 (Function Cheat Sheet)

| 函数 | 作用 (大白话) | 常用参数 | SQL 类比 |
| :--- | :--- | :--- | :--- |
| `pd.merge()` | 多表拼接 | `on='store_nbr'`, `how='left'` | `LEFT JOIN` |
| `pd.to_datetime()` | 字符串 → 时间 | `format='%Y-%m-%d'` | `CAST(col AS DATE)` |
| `df.groupby().transform()` | 分组内计算 (保留原行数) | `lambda x: x.shift(7)` | `LAG() OVER(PARTITION BY)` |
| `df.pivot_table()` | 长表变宽表 | `index, columns, values` | `PIVOT` |
| `pd.get_dummies()` | One-Hot 编码 | `drop_first=True` | `CASE WHEN` |

> **⚠️ 防泄露铁律**: `shift()` 参数 ≥ 预测步长！
> `groupby(['store_nbr','family']).shift(7)` = 按门店×品类取上周同天值。

## 1. 数据导入 (Data Loading)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.style.use('seaborn-v0_8-whitegrid')

# 数据路径
DATA_DIR = 'data/store_sales/'

# 加载所有表
train = pd.read_csv(f'{DATA_DIR}train.csv', parse_dates=['date'])
stores = pd.read_csv(f'{DATA_DIR}stores.csv')
oil = pd.read_csv(f'{DATA_DIR}oil.csv', parse_dates=['date'])
holidays = pd.read_csv(f'{DATA_DIR}holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(f'{DATA_DIR}transactions.csv', parse_dates=['date'])

print(f'Train shape: {train.shape}')
train.head()

Train shape: (3000888, 6)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


---

## 2. 数据准备 (Data Preparation)
> **任务**: 
> 1. 合并多表 (train + stores + oil + holidays)
> 2. 检查缺失值 & 数据类型
> 3. 快速可视化：总销量时间序列

**🤔 思考**: 
- `oil.csv` 有缺失值（非交易日无油价），怎么处理？
- `holidays` 表有 `transferred` 标记，是什么意思？
- 合并后数据量会不会爆炸？需不需要先筛选部分门店？

In [5]:
# ✅ 正确做法：用字典把名字和变量对应起来
sheets = {
    'train': train,
    'oil': oil,
    'stores': stores,
    'transactions': transactions,
    'holidays': holidays

}

for name, df in sheets.items():
    print(f'=== {name} === shape: {df.shape}')
    display(df.head(2))    # notebook 里用 display 比 print 更好看
    print()


=== train === shape: (3000888, 6)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0



=== oil === shape: (1218, 2)


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14



=== stores === shape: (54, 5)


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13



=== transactions === shape: (83488, 3)


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111



=== holidays === shape: (350, 6)


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False


In [11]:
holidays.describe(include='all')


,date,type,locale,locale_name,description,transferred
count,350,350,350,350,350,350
unique,NaN,6,3,24,103,2
top,NaN,Holiday,National,Ecuador,Carnaval,False
freq,NaN,221,174,174,10,338
mean,2015-04-24 00:45:15.428571392,NaN,NaN,NaN,NaN,NaN
min,2012-03-02 00:00:00,NaN,NaN,NaN,NaN,NaN
25%,2013-12-23 06:00:00,NaN,NaN,NaN,NaN,NaN
50%,2015-06-08 00:00:00,NaN,NaN,NaN,NaN,NaN
75%,2016-07-03 00:00:00,NaN,NaN,NaN,NaN,NaN
max,2017-12-26 00:00:00,NaN,NaN,NaN,NaN,NaN


In [ ]:
details = pd.merge(train,stores,how='left',on='store_nbr')
details = pd.merge(details,holidays)

---

## 3. 探索性数据分析 (EDA)
> **任务**:
> 1. 按 `family` (品类) 看销量分布 → 哪些品类销量最高？
> 2. 按 `store_nbr`/`type` 看门店差异
> 3. 时间维度：年/月/周 趋势 + 周期性
> 4. 促销 (`onpromotion`) 对销量的影响
> 5. 油价 (`dcoilwtico`) 与总销量的相关性

**🤔 思考**: 
- 不同品类的销售模式可能完全不同（日用品 vs 奢侈品），需要分开看
- 节假日前后的销量变化是否明显？

---

## 4. 特征工程 (Feature Engineering)
> **任务**:
> 1. **日期特征**: year, month, dayofweek, is_weekend, is_month_start/end
> 2. **Per-Category 时序特征** ⭐ (按 store_nbr × family 分组):
>    - Lag: `lag7`, `lag14`, `lag28`, `lag364`
>    - Diff: `diff7`, `diff364`
>    - Rolling: `rolling_mean_7`, `rolling_std_7`, `rolling_mean_28`
> 3. **促销特征**: `onpromotion` (已有) + 促销天数累计
> 4. **外部特征**: 油价 (可滞后) + 节假日标记
> 5. **门店特征**: `type`, `cluster` (One-Hot 编码)

**💡 关键代码模式**:
```python
# Per-Category Lag (按 门店×品类 分组取滞后值)
df['sales_lag7'] = df.groupby(['store_nbr', 'family'])['sales'].shift(7)
```

**⚠️ 注意**:
- Lag/Rolling 会产生 NaN → 训练前 `dropna()`
- 检查 Feature Importance 防 Data Leakage！

---

## 5. 数据切分 (Train/Test Split)
> **任务**:
> 1. 按时间切分 (**不能随机切！**)
> 2. 建议：最后 30 天作为测试集
> 3. 检查 Train/Test 分布是否一致

```python
# 示例
split_date = '2017-07-15'
train_mask = df['date'] < split_date
```

---

## 6. 建模 (Modeling)
> **任务**:
> 1. XGBoost + Early Stopping
> 2. 训练后**立刻**看 Feature Importance → 检查 Data Leakage
> 3. 预测 & 计算 M[目标业务]

**🤔 思考**: 
- 特征数量可能很多，注意 `colsample_bytree` 参数
- 3M 行数据训练可能较慢，可先用 1-2 个门店调试

---

## 7. 评估与对比 (Evaluation)
> **任务**:
> 1. M[目标业务] / Mean 比率 (快速诊断)
> 2. **分层评估**: Normal Days vs Extreme Days
> 3. **按品类 M[目标业务]**: 哪些品类好预测？哪些难？
> 4. 预测 vs 真实值折线图
> 5. 残差分析 (残差分布 + 按时间维度拆解)

---

## 8. 结论与建议 (Insights)
> **任务**: 用 3 句话总结你的发现：
> 1. 模型整体表现如何？(M[目标业务]/Mean)
> 2. 最大的发现是什么？(Feature Importance / 品类差异)
> 3. 如果要继续优化，下一步做什么？